# DFlash与DSpark数值案例与vLLM测试

本notebook分两章：

* 第1章用小矩阵把一轮投机解码数据流跑通（不加载真实LLM）
* 第2章给出vLLM上的下载、评估与部署测试步骤

相关文章链接：[快速理解并行投机解码(DFlash/DSpark)](https://zhuanlan.zhihu.com/p/2069029506447417522)

Author: kaiyuan

Email: kaiyuanxie@yeah.net


# 1 原理数值演示

## 1.1 背景速览

两个计算模式：

* 词语接龙：causal mask，自回归挨个吐字（主模型decoding、以及EAGLE这类自回归草稿）
* 完型填空：可读全句、一次填多空（DFlash草稿的并行block）

投机解码：小模型先猜一截，主模型再校验。DFlash把草稿改成完型填空，并用主模型中间层融成$H_{ctx}$注入Draft；DSpark再在骨干后加串行头与Hardware-Aware Prefix Scheduler。

下面从1.2节起逐步做数值演示。


## 1.2 准备工作

初始化词表与随机权重。词表用“我是”+字母位，方便人眼读序列。


In [1]:
import math
import numpy as np

# 词表：前缀“我是”+后续英文字母位
VOCAB = ["<pad>", "<mask>", "我", "是", "k", "a", "i", "y"]
MASK_ID = 1
VOCAB_SIZE = len(VOCAB)
D_MODEL = 4
N_DRAFT_LAYERS = 2
BLOCK_SIZE = 4
GAMMA = BLOCK_SIZE - 1  # DFlash默认只预测mask位
MARKOV_RANK = 2
CTX_LAYER_IDS = [1, 2]

rng = np.random.default_rng(42)

def softmax(x, axis=-1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def rms_norm(x, eps=1e-6):
    return x / np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)

def tokens_to_str(ids):
    return "".join(VOCAB[i] for i in ids)

def show(name, arr, meaning):
    a = np.asarray(arr)
    print(f"[{name}] shape={a.shape}  # {meaning}")
    print(np.round(a, 4))
    print()

def init_linear(in_f, out_f, scale=0.3):
    return rng.normal(0, scale, size=(out_f, in_f))

# Embedding / LM head 与主模型共享（演示里用同一组矩阵）
E = rng.normal(0, 0.4, size=(VOCAB_SIZE, D_MODEL))
E[MASK_ID] = rng.normal(0, 0.05, size=(D_MODEL,))
W_lm = E.copy()
W_c = init_linear(len(CTX_LAYER_IDS) * D_MODEL, D_MODEL, 0.25)

draft_layers = []
for _ in range(N_DRAFT_LAYERS):
    draft_layers.append({
        "Wq": init_linear(D_MODEL, D_MODEL),
        "Wk": init_linear(D_MODEL, D_MODEL),
        "Wv": init_linear(D_MODEL, D_MODEL),
        "Wo": init_linear(D_MODEL, D_MODEL),
        "W1": init_linear(D_MODEL, D_MODEL * 2),
        "W2": init_linear(D_MODEL * 2, D_MODEL),
    })

_TARGET_TRANSFORMS = [init_linear(D_MODEL, D_MODEL, 0.2) for _ in range(4)]
W_markov_emb = rng.normal(0, 0.3, size=(VOCAB_SIZE, MARKOV_RANK))
W_markov_proj = rng.normal(0, 0.3, size=(MARKOV_RANK, VOCAB_SIZE))
w_conf = rng.normal(0, 0.4, size=(D_MODEL + MARKOV_RANK,))

print(f"就绪 | vocab={VOCAB}")
print(f"D={D_MODEL}, draft_layers={N_DRAFT_LAYERS}, block_size={BLOCK_SIZE}, gamma={GAMMA}")


就绪 | vocab=['<pad>', '<mask>', '我', '是', 'k', 'a', 'i', 'y']
D=4, draft_layers=2, block_size=4, gamma=3


In [2]:
def target_forward(token_ids):
    """模拟主模型forward：返回末位置logits + 各层hidden。"""
    h = E[np.asarray(token_ids)].copy()
    hiddens = []
    for li, Wt in enumerate(_TARGET_TRANSFORMS):
        h = rms_norm(h + 0.15 * (li + 1) * np.tanh(h @ Wt.T))
        hiddens.append(h.copy())
    logits = h[-1] @ W_lm.T
    return logits, hiddens

def fuse_target_context(hiddens, positions=None):
    """H_ctx = RMSNorm(W_c [H^{l1}; ...; H^{lm}])"""
    if positions is None:
        positions = list(range(hiddens[0].shape[0]))
    feats = [np.concatenate([hiddens[li][p] for li in CTX_LAYER_IDS], axis=-1) for p in positions]
    return rms_norm(np.stack(feats, axis=0) @ W_c.T)

def bidirectional_attn(Q, K, V):
    scale = 1.0 / math.sqrt(Q.shape[-1])
    weights = softmax((Q @ K.T) * scale, axis=-1)
    return weights @ V, weights

def draft_layer_forward(H_d, H_ctx, layer):
    """KV injection: K/V=[proj(H_ctx); proj(H_d)], Q=proj(H_d)"""
    Q = H_d @ layer["Wq"].T
    K = np.concatenate([H_ctx @ layer["Wk"].T, H_d @ layer["Wk"].T], axis=0)
    V = np.concatenate([H_ctx @ layer["Wv"].T, H_d @ layer["Wv"].T], axis=0)
    out, attn_w = bidirectional_attn(Q, K, V)
    H = rms_norm(H_d + out @ layer["Wo"].T)
    ff = np.maximum(0, H @ layer["W1"].T) @ layer["W2"].T
    H = rms_norm(H + ff)
    return H, attn_w, (Q.shape, K.shape, V.shape)

print("已定义 target_forward / fuse_target_context / draft_layer_forward")


已定义 target_forward / fuse_target_context / draft_layer_forward


## 1.3 DFlash一轮

拆成Step A/B/C：

1. 主模型根据前缀写出anchor，并抽出中间层hidden融成$H_{ctx}$
2. Draft输入$[\mathrm{anchor},\langle m\rangle,\ldots]$，Attention里K/V为$[H_{ctx}\,||\,H_d]$，Query只来自$H_d$；一次forward填完mask
3. 主模型校验草稿最长前缀，再写出新的bonus

**Step A：anchor与$H_{ctx}$**


In [3]:
# Step A：主模型产出 anchor 与 H_ctx
prompt = [2, 3]  # 我是
print(f"前缀: {prompt} -> {tokens_to_str(prompt)}")

logits, hiddens = target_forward(prompt)
show("target_logits", logits, "末位置词表打分")
bonus = int(np.argmax(logits))
print(f"argmax得到anchor/bonus = {bonus} ({VOCAB[bonus]})")
print()

for li in CTX_LAYER_IDS:
    show(f"H_layer_{li}", hiddens[li], "主模型中间层hidden")

H_ctx = fuse_target_context(hiddens)
show("H_ctx", H_ctx, "跨层拼接+投影+RMSNorm后的上下文，稍后注入Draft的K/V")

prefix = prompt + [bonus]
print(f"更新前缀: {prefix} -> {tokens_to_str(prefix)}")


前缀: [2, 3] -> 我是
[target_logits] shape=(8,)  # 末位置词表打分
[-0.6551 -0.0853 -0.4317  1.1611 -0.1015  0.2293 -0.129   1.2854]

argmax得到anchor/bonus = 7 (y)

[H_layer_1] shape=(2, 4)  # 主模型中间层hidden
[[-0.083  -0.9978  1.2457  1.2025]
 [ 0.0965  1.4313  0.7725 -1.1599]]

[H_layer_2] shape=(2, 4)  # 主模型中间层hidden
[[-0.2265 -1.0761  1.021   1.3222]
 [ 0.0788  1.4579  0.8699 -1.0544]]

[H_ctx] shape=(2, 4)  # 跨层拼接+投影+RMSNorm后的上下文，稍后注入Draft的K/V
[[-0.3479  1.352  -1.3751 -0.4002]
 [-1.9572  0.3613  0.1239  0.1525]]

更新前缀: [2, 3, 7] -> 我是y


**Step B：Draft并行填空**

打印内容：

* `H_d_input`：只有anchor是真token，后面是mask
* 每层`K/V`长度 = 上下文长度 + block长度，即把$H_{ctx}$拼进K/V前半段
* `mask_logits`：一次得到，位置之间没有词语接龙依赖


In [4]:
# Step B：草稿模型一次并行填空（完型填空，不是词语接龙）
print(f"anchor={bonus}({VOCAB[bonus]}), gamma={GAMMA}, mask={MASK_ID}")
print()

H_ctx_last = H_ctx[-1:]
show("H_ctx_for_draft", H_ctx_last, "本轮注入Draft的上下文（演示取末位置）")

input_ids = [bonus] + [MASK_ID] * GAMMA
H_d = E[np.asarray(input_ids)].copy()
print(f"Draft输入: {input_ids} -> {tokens_to_str(input_ids)}")
show("H_d_input", H_d, "第0位是干净anchor，后面是mask占位")

for li, layer in enumerate(draft_layers):
    H_d, attn_w, (qshape, kshape, vshape) = draft_layer_forward(H_d, H_ctx_last, layer)
    print(f"--- Draft Layer {li} ---")
    print(f"Q={qshape}（只来自Draft）; K/V={kshape}（前半H_ctx，后半H_d）")
    show(f"attn_L{li}", attn_w, "Draft query对[H_ctx|H_d]的注意力（无causal mask）")
    show(f"H_d_L{li}", H_d, "该层输出hidden")

logits_all = H_d @ W_lm.T
mask_logits = logits_all[1:]
base_probs = softmax(mask_logits, axis=-1)
draft_ids = [int(np.argmax(base_probs[k])) for k in range(GAMMA)]

show("mask_logits", mask_logits, "同一次forward得到的gamma个并行预测")
print("草稿token:")
for k, tid in enumerate(draft_ids):
    print(f"  pos{k+1}: {tid}({VOCAB[tid]}) p={base_probs[k][tid]:.4f}")
print(f"draft = {draft_ids} -> {tokens_to_str(draft_ids)}")


anchor=7(y), gamma=3, mask=1

[H_ctx_for_draft] shape=(1, 4)  # 本轮注入Draft的上下文（演示取末位置）
[[-1.9572  0.3613  0.1239  0.1525]]

Draft输入: [7, 1, 1, 1] -> y<mask><mask><mask>
[H_d_input] shape=(4, 4)  # 第0位是干净anchor，后面是mask占位
[[ 0.1651  0.1723  0.8567 -0.1626]
 [-0.0256 -0.0407  0.0308  0.0564]
 [-0.0256 -0.0407  0.0308  0.0564]
 [-0.0256 -0.0407  0.0308  0.0564]]

--- Draft Layer 0 ---
Q=(4, 4)（只来自Draft）; K/V=(5, 4)（前半H_ctx，后半H_d）
[attn_L0] shape=(4, 5)  # Draft query对[H_ctx|H_d]的注意力（无causal mask）
[[0.1846 0.2019 0.2045 0.2045 0.2045]
 [0.1973 0.2013 0.2005 0.2005 0.2005]
 [0.1973 0.2013 0.2005 0.2005 0.2005]
 [0.1973 0.2013 0.2005 0.2005 0.2005]]

[H_d_L0] shape=(4, 4)  # 该层输出hidden
[[ 0.2298  0.2692  1.9323 -0.3753]
 [-1.2054 -0.9521  0.3841  1.2218]
 [-1.2054 -0.9521  0.3841  1.2218]
 [-1.2054 -0.9521  0.3841  1.2218]]

--- Draft Layer 1 ---
Q=(4, 4)（只来自Draft）; K/V=(5, 4)（前半H_ctx，后半H_d）
[attn_L1] shape=(4, 5)  # Draft query对[H_ctx|H_d]的注意力（无causal mask）
[[0.1551 0.2136 0.2104 0.2104 0.210

**Step C：主模型校验并写bonus**

与常规投机推理相同：从左核对到第一处不一致为止；该位起草稿作废，由主模型给出bonus（下轮的anchor）。


In [5]:
# Step C：主模型校验（greedy最长前缀）+ 写出新bonus
print(f"校验前前缀: {tokens_to_str(prefix)}")
print(f"待校验草稿: {tokens_to_str(draft_ids)}")
print()

accepted = []
cur = list(prefix)
bonus_new = None
for i, d in enumerate(draft_ids):
    logits_v, _ = target_forward(cur)
    probs_v = softmax(logits_v)
    pred = int(np.argmax(logits_v))
    ok = pred == d
    print(
        f"位{i}: draft={d}({VOCAB[d]}) target={pred}({VOCAB[pred]}) "
        f"p_t={probs_v[d]:.4f}  {'ACCEPT' if ok else 'REJECT'}"
    )
    if not ok:
        bonus_new = pred
        print(f"  从该位起丢弃；主模型写出bonus={bonus_new}({VOCAB[bonus_new]})")
        break
    accepted.append(d)
    cur.append(d)
else:
    logits_v, _ = target_forward(cur)
    bonus_new = int(np.argmax(logits_v))
    print(f"全部接受；主模型再写bonus={bonus_new}({VOCAB[bonus_new]})")

new_prefix = prefix + accepted + [bonus_new]
print()
print(f"接受草稿长度={len(accepted)}/{len(draft_ids)}（另外写入1个bonus）")
print(f"本轮后序列: {tokens_to_str(new_prefix)}")


校验前前缀: 我是y
待校验草稿: 我我我

位0: draft=2(我) target=7(y) p_t=0.0834  REJECT
  从该位起丢弃；主模型写出bonus=7(y)

接受草稿长度=0/3（另外写入1个bonus）
本轮后序列: 我是yy


## 1.4 DSpark：在DFlash骨干上多两段

parallel block仍是DFlash式完型填空；之后加sequential block（Markov/RNN）与筛选。

**Step D：并行logits + 串行修正 + confidence**

先一次算出各位置$U$，再从左到右用上一token的转移偏置$B$修正后采样，并估计条件置信度$c_k$。


In [6]:
# Step D：DSpark = 并行骨干 + Markov串行头 + confidence
print("D-1 并行骨干：一次forward得到各位置base logits U")

def markov_bias(prev_id):
    return W_markov_emb[prev_id] @ W_markov_proj

gamma_ds = BLOCK_SIZE
input_ids_ds = [bonus] + [MASK_ID] * (gamma_ds - 1)
H_d = E[np.asarray(input_ids_ds)].copy()
for layer in draft_layers:
    H_d, _, _ = draft_layer_forward(H_d, H_ctx_last, layer)

U = H_d @ W_lm.T
show("U", U, "并行骨干打分（此时块内还没串起来）")

print("D-2 从左到右：U + Markov偏置B，采样并打confidence")
draft_ids_ds, confs = [], []
prev = bonus
for k in range(gamma_ds):
    B = markov_bias(prev)
    logits_k = U[k] + B
    probs_k = softmax(logits_k)
    tid = int(np.argmax(probs_k))
    feat = np.concatenate([H_d[k], W_markov_emb[prev]])
    c = 1.0 / (1.0 + math.exp(-float(w_conf @ feat)))
    print(f"k={k+1} prev={prev}({VOCAB[prev]}) -> {tid}({VOCAB[tid]}) p={probs_k[tid]:.4f} c={c:.4f}")
    draft_ids_ds.append(tid)
    confs.append(c)
    prev = tid

print(f"草稿: {tokens_to_str(draft_ids_ds)}")
print(f"confidence: {[round(c, 4) for c in confs]}")


D-1 并行骨干：一次forward得到各位置base logits U
[U] shape=(4, 8)  # 并行骨干打分（此时块内还没串起来）
[[ 0.449   0.0403  0.5761  0.4998  0.6865  0.9769  0.3663  1.7476]
 [ 0.8651  0.1535  0.9123 -0.6662  0.3508  0.5692  0.642   0.1   ]
 [ 0.8651  0.1535  0.9123 -0.6662  0.3508  0.5692  0.642   0.1   ]
 [ 0.8651  0.1535  0.9123 -0.6662  0.3508  0.5692  0.642   0.1   ]]

D-2 从左到右：U + Markov偏置B，采样并打confidence
k=1 prev=7(y) -> 7(y) p=0.2872 c=0.4143
k=2 prev=7(y) -> 0(<pad>) p=0.2110 c=0.3149
k=3 prev=0(<pad>) -> 0(<pad>) p=0.2166 c=0.3373
k=4 prev=0(<pad>) -> 0(<pad>) p=0.2166 c=0.3373
草稿: y<pad><pad><pad>
confidence: [0.4143, 0.3149, 0.3373, 0.3373]


**Step E：按“值不值得验”裁剪长度**

Hardware-Aware Prefix Scheduler大致是：

* $a_j=\prod_{i\le j}c_i$
* 用profile好的$\mathrm{SPS}(B)$估$\Theta=\tau\cdot\mathrm{SPS}(B)$
* 贪心加长验证前缀，吞吐不再升就停，只把前$\ell^*$个送给主模型


In [7]:
# Step E：Hardware-Aware Prefix Scheduler
print("E-1 前缀存活概率 a_j = Π c_i")
SPS = {1: 100.0, 2: 95.0, 3: 88.0, 4: 78.0, 5: 65.0, 6: 50.0, 7: 38.0, 8: 28.0}
a, prod = [], 1.0
for c in confs:
    prod *= c
    a.append(prod)
print(f"c={ [round(x,4) for x in confs] }")
print(f"a={ [round(x,4) for x in a] }")

print("E-2 按SPS(B)扩展验证长度，Theta不再升就停")
R = 1
best_theta = R * SPS[R]
best_ell, tau, Bsz = 0, float(R), R
for j, aj in enumerate(a, start=1):
    B_new = Bsz + 1
    tau_new = tau + aj
    theta = tau_new * SPS.get(B_new, SPS[max(SPS)])
    print(f"  ell={j}: B={B_new}, 增量a={aj:.4f}, Theta={theta:.2f}")
    if theta > best_theta:
        best_theta, best_ell, tau, Bsz = theta, j, tau_new, B_new
    else:
        print(f"  停在 ell*={best_ell}")
        break

scheduled = draft_ids_ds[:best_ell]
print(f"ell*={best_ell}, 送去校验: {tokens_to_str(scheduled)}")

print("E-3 主模型只校验该前缀")
accepted_ds, cur = [], list(prefix)
bonus_ds = None
for i, d in enumerate(scheduled):
    logits_v, _ = target_forward(cur)
    pred = int(np.argmax(logits_v))
    ok = pred == d
    print(f"  位{i}: draft={d}({VOCAB[d]}) target={pred}({VOCAB[pred]}) {'ACCEPT' if ok else 'REJECT'}")
    if not ok:
        bonus_ds = pred
        break
    accepted_ds.append(d)
    cur.append(d)
else:
    logits_v, _ = target_forward(cur if scheduled else prefix)
    bonus_ds = int(np.argmax(logits_v))

print(f"结果: accepted={tokens_to_str(accepted_ds)}, bonus={bonus_ds}({VOCAB[bonus_ds]})")


E-1 前缀存活概率 a_j = Π c_i
c=[0.4143, 0.3149, 0.3373, 0.3373]
a=[0.4143, 0.1305, 0.044, 0.0148]
E-2 按SPS(B)扩展验证长度，Theta不再升就停
  ell=1: B=2, 增量a=0.4143, Theta=134.36
  ell=2: B=3, 增量a=0.1305, Theta=135.94
  ell=3: B=4, 增量a=0.0440, Theta=123.92
  停在 ell*=2
ell*=2, 送去校验: y<pad>
E-3 主模型只校验该前缀
  位0: draft=7(y) target=7(y) ACCEPT
  位1: draft=0(<pad>) target=7(y) REJECT
结果: accepted=y, bonus=7(y)


## 1.5 符号速查

| 符号 | 含义 |
|---|---|
| Target / Draft | 主模型 / 草稿模型 |
| anchor / bonus | 主模型写出的干净token：进本轮输出，并作下轮Draft条件 |
| $H_{ctx}$ | 主模型多层hidden融合后的上下文 |
| KV injection | 把$H_{ctx}$拼进Draft每层K/V |
| $\gamma$ | 草稿token数；DFlash常为block_size-1 |
| $U$ / $B$ | 并行骨干logits / Markov转移偏置 |
| $c$ / $a$ / $\ell^*$ | 条件置信度 / 前缀存活概率 / 实际验证长度 |
| $\mathrm{SPS}(B)$ | 引擎在batch大小为$B$时的Steps Per Second |


# 2 vLLM测试


## 2.1 数据下载

```
# 1. 安装依赖
pip install datasets

# 2. 下载全部数据集 (默认输出到 ./eval_data/)
python3 download_eval_data.py

# 3. 指定输出目录和样本数
python3 download_eval_data.py --output-dir /root/eval_data --num-samples 250

# 4. 只下载某个数据集
python3 download_eval_data.py --dataset gsm8k
```

download_eval_data.py 脚本：


In [ ]:
#!/usr/bin/env python3
"""
评估数据集下载与预处理脚本 (参考)

数据源: HuggingFace
- GSM8K: openai/gsm8k (test split, 1319 题) → 随机抽取 250 题
- MMLU:  cais/mmlu      (test split, 14042 题) → 随机抽取 250 题
- HumanEval: openai/openai_humaneval (test, 164 题) → 全量

输出 JSON 格式 (与 eval_reference_script.py 兼容):
  gsm8k_test_250.json:    [{"question": "...", "answer": "..."}, ...]
  mmlu_test_250.json:     [{"question": "...", "choices": [...], "answer": 0, "subject": "..."}, ...]
  humaneval_test_164.json:[{"task_id": "...", "prompt": "...", "test": "...", "entry_point": "..."}, ...]

依赖安装:
    pip install datasets

使用方法:
    # 下载全部数据集 (默认输出到 ./eval_data/)
    python3 download_eval_data.py

    # 指定输出目录和样本数
    python3 download_eval_data.py --output-dir /root/eval_data --num-samples 250

    # 只下载某个数据集
    python3 download_eval_data.py --dataset gsm8k

    # 使用 HuggingFace 镜像 (国内网络)
    HF_ENDPOINT=https://hf-mirror.com python3 download_eval_data.py

注意:
    - 服务器若无法直连 HuggingFace, 需设置代理或镜像
    - 国内推荐: export HF_ENDPOINT=https://hf-mirror.com
    - 或使用代理: export HTTPS_PROXY=http://10.173.103.37:3128
    - 若下载 xet 格式报错, 设置: export HF_HUB_DISABLE_XET=1
"""

import argparse
import json
import os
import random
import sys
from typing import List, Dict


def download_gsm8k(num_samples: int = 250, seed: int = 42) -> List[Dict]:
    """
    下载 GSM8K 数据集
    来源: openai/gsm8k, test split
    格式: {"question": "...", "answer": "..."}
    """
    from datasets import load_dataset

    print("[GSM8K] 加载 openai/gsm8k ...")
    ds = load_dataset("openai/gsm8k", "main", split="test")
    print(f"[GSM8K] 总样本数: {len(ds)}")

    # 随机抽样
    random.seed(seed)
    indices = list(range(len(ds)))
    random.shuffle(indices)
    indices = indices[:num_samples]

    samples = []
    for idx in indices:
        item = ds[idx]
        # GSM8K answer 字段格式: "<<18>> 18" → 提取数值
        answer_raw = item["answer"]
        # 提取 #### 后的数值
        if "####" in answer_raw:
            answer = answer_raw.split("####")[-1].strip()
        else:
            # 格式: "<<18>> 18" → 取最后一段
            answer = answer_raw.split(">>")[-1].strip()

        samples.append({
            "question": item["question"],
            "answer": answer,
        })

    print(f"[GSM8K] 抽取样本: {len(samples)}")
    return samples


def download_mmlu(num_samples: int = 250, seed: int = 42) -> List[Dict]:
    """
    下载 MMLU 数据集
    来源: cais/mmlu, test split (all subjects)
    格式: {"question": "...", "choices": [4 options], "answer": 0-3, "subject": "..."}
    """
    from datasets import load_dataset

    print("[MMLU] 加载 cais/mmlu (all_config) ...")
    # cais/mmlu 使用 'all' 配置加载所有科目
    ds = load_dataset("cais/mmlu", "all", split="test")
    print(f"[MMLU] 总样本数: {len(ds)}")

    # 随机抽样
    random.seed(seed)
    indices = list(range(len(ds)))
    random.shuffle(indices)
    indices = indices[:num_samples]

    samples = []
    for idx in indices:
        item = ds[idx]
        samples.append({
            "question": item["question"],
            "choices": item["choices"],
            "answer": item["answer"],  # 0=A, 1=B, 2=C, 3=D
            "subject": item.get("subject", "unknown"),
        })

    print(f"[MMLU] 抽取样本: {len(samples)}")
    return samples


def download_humaneval() -> List[Dict]:
    """
    下载 HumanEval 数据集
    来源: openai/openai_humaneval, test split (全量 164 题)
    格式: {"task_id", "prompt", "canonical_solution", "test", "entry_point"}
    """
    from datasets import load_dataset

    print("[HumanEval] 加载 openai/openai_humaneval ...")
    ds = load_dataset("openai/openai_humaneval", split="test")
    print(f"[HumanEval] 总样本数: {len(ds)}")

    samples = []
    for item in ds:
        samples.append({
            "task_id": item["task_id"],
            "prompt": item["prompt"],
            "canonical_solution": item["canonical_solution"],
            "test": item["test"],
            "entry_point": item["entry_point"],
        })

    print(f"[HumanEval] 全量样本: {len(samples)}")
    return samples


def save_json(samples: List[Dict], filepath: str):
    """保存为 JSON 文件"""
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(samples, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  已保存: {filepath} ({len(samples)} 样本, {size_kb:.1f} KB)")


def verify_samples(samples: List[Dict], dataset_name: str):
    """验证样本格式"""
    if not samples:
        print(f"  [ERROR] {dataset_name}: 空数据")
        return False

    required_fields = {
        "gsm8k": ["question", "answer"],
        "mmlu": ["question", "choices", "answer"],
        "humaneval": ["task_id", "prompt", "test", "entry_point"],
    }

    fields = required_fields.get(dataset_name, [])
    for i, s in enumerate(samples[:3]):  # 检查前 3 个
        for f in fields:
            if f not in s:
                print(f"  [ERROR] {dataset_name}[{i}]: 缺少字段 {f}")
                return False

    print(f"  [OK] {dataset_name}: 格式验证通过 (前3样本)")
    return True


def main():
    parser = argparse.ArgumentParser(description="评估数据集下载脚本")
    parser.add_argument(
        "--output-dir", default="./eval_data",
        help="输出目录 (默认 ./eval_data)",
    )
    parser.add_argument(
        "--num-samples", type=int, default=250,
        help="GSM8K/MMLU 抽样数 (默认 250, HumanEval 全量)",
    )
    parser.add_argument(
        "--dataset", default="all",
        choices=["all", "gsm8k", "mmlu", "humaneval"],
        help="下载哪个数据集 (默认 all)",
    )
    parser.add_argument("--seed", type=int, default=42, help="随机种子")
    args = parser.parse_args()

    print(f"{'=' * 60}")
    print(f"评估数据集下载")
    print(f"{'=' * 60}")
    print(f"输出目录: {args.output_dir}")
    print(f"样本数: {args.num_samples} (HumanEval 全量)")
    print(f"随机种子: {args.seed}")
    print(f"HF_ENDPOINT: {os.environ.get('HF_ENDPOINT', '默认 (https://huggingface.co)')}")
    print()

    # 检查 datasets 库
    try:
        import datasets
        print(f"datasets 版本: {datasets.__version__}")
    except ImportError:
        print("[ERROR] 需要安装 datasets 库:")
        print("  pip install datasets")
        sys.exit(1)

    # 下载数据
    if args.dataset in ("all", "gsm8k"):
        try:
            samples = download_gsm8k(args.num_samples, args.seed)
            if verify_samples(samples, "gsm8k"):
                save_json(samples, os.path.join(args.output_dir, "gsm8k_test_250.json"))
        except Exception as e:
            print(f"[GSM8K] 下载失败: {e}")

    if args.dataset in ("all", "mmlu"):
        try:
            samples = download_mmlu(args.num_samples, args.seed)
            if verify_samples(samples, "mmlu"):
                save_json(samples, os.path.join(args.output_dir, "mmlu_test_250.json"))
        except Exception as e:
            print(f"[MMLU] 下载失败: {e}")

    if args.dataset in ("all", "humaneval"):
        try:
            samples = download_humaneval()
            if verify_samples(samples, "humaneval"):
                save_json(samples, os.path.join(args.output_dir, "humaneval_test_164.json"))
        except Exception as e:
            print(f"[HumanEval] 下载失败: {e}")

    print(f"\n{'=' * 60}")
    print(f"下载完成")
    print(f"{'=' * 60}")
    print(f"\n下一步: 使用 eval_reference_script.py 运行评估")
    print(f"  python3 eval_reference_script.py --data-dir {args.output_dir} --test-throughput")


if __name__ == "__main__":
    main()


## 2.2 评估脚本

eval_reference_script.py


In [ ]:
#!/usr/bin/env python3
"""
推测解码评估参考脚本 (Speculative Decoding Evaluation Reference)

功能：
1. 精度评估: GSM8K (数学推理) / MMLU (多任务理解) / HumanEval (代码生成)
2. 吞吐量测试: 固定 prompt 循环请求，计算 tok/s
3. 延迟测试: TTFT (首 token 延迟) / 平均延迟 / P50 / P95
4. 多服务对比: DSpark / DFlash / Baseline 同时评估
5. 报告生成: JSON 详细报告 + Markdown 汇总表

依赖: 仅标准库 (urllib, json, time, concurrent.futures)

使用方法:
    # 基础用法: DSpark vs DFlash 精度+吞吐量
    python3 eval_reference_script.py \\
        --dspark-url http://127.0.0.1:8000 \\
        --dflash-url http://127.0.0.1:8001 \\
        --data-dir ./eval_data \\
        --test-throughput

    # 加入 Baseline 对照组
    python3 eval_reference_script.py \\
        --dspark-url http://127.0.0.1:8000 \\
        --dflash-url http://127.0.0.1:8001 \\
        --baseline-url http://127.0.0.1:8002 \\
        --data-dir ./eval_data \\
        --test-throughput

    # 只测某个数据集 + 限制样本数
    python3 eval_reference_script.py \\
        --dspark-url http://127.0.0.1:8000 \\
        --data-dir ./eval_data \\
        --dataset gsm8k \\
        --num-samples 50
"""

import argparse
import json
import time
import sys
import os
import re
import statistics
import urllib.request
import urllib.error
from typing import Optional, List, Dict, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================
# HTTP 客户端
# ============================================================

class LLMClient:
    """LLM 服务客户端，封装请求逻辑"""

    def __init__(self, base_url: str, name: str, timeout: int = 180):
        self.base_url = base_url.rstrip("/")
        self.name = name
        self.timeout = timeout
        self.model = self._discover_model()

    def _discover_model(self) -> Optional[str]:
        """自动发现可用模型"""
        try:
            req = urllib.request.Request(
                f"{self.base_url}/v1/models",
                headers={"User-Agent": "eval_ref/1.0"},
            )
            with urllib.request.urlopen(req, timeout=10) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                models = data.get("data", [])
                return models[0].get("id") if models else None
        except Exception as e:
            print(f"[WARN] {self.name}: 无法获取模型列表: {e}")
            return None

    def chat(
        self,
        messages: List[Dict],
        max_tokens: int = 512,
        temperature: float = 0.0,
        top_p: float = 1.0,
        stream: bool = False,
    ) -> Tuple[dict, float, Optional[float]]:
        """
        发送 chat completion 请求
        返回: (response, total_latency, ttft)
          - ttft: Time To First Token (仅 stream=True 时有效)
        """
        payload = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p,
            "stream": stream,
        }
        data = json.dumps(payload).encode("utf-8")
        req = urllib.request.Request(
            f"{self.base_url}/v1/chat/completions",
            data=data,
            headers={
                "Content-Type": "application/json",
                "User-Agent": "eval_ref/1.0",
            },
            method="POST",
        )

        try:
            t0 = time.time()
            if not stream:
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    result = json.loads(resp.read().decode("utf-8"))
                    latency = time.time() - t0
                    return result, latency, None
            else:
                # 流式请求: 测量 TTFT
                ttft = None
                collected_content = []
                usage = {}
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    for line in resp:
                        line = line.decode("utf-8").strip()
                        if not line or not line.startswith("data: "):
                            continue
                        chunk = line[6:]
                        if chunk == "[DONE]":
                            break
                        try:
                            chunk_data = json.loads(chunk)
                        except json.JSONDecodeError:
                            continue
                        if ttft is None and chunk_data.get("choices", [{}])[0].get("delta", {}).get("content"):
                            ttft = time.time() - t0
                        delta = chunk_data.get("choices", [{}])[0].get("delta", {})
                        if "content" in delta:
                            collected_content.append(delta["content"])
                        if "usage" in chunk_data:
                            usage = chunk_data["usage"]

                latency = time.time() - t0
                result = {
                    "choices": [{"message": {"content": "".join(collected_content)}}],
                    "usage": usage,
                }
                return result, latency, ttft
        except urllib.error.HTTPError as e:
            return {"error": f"HTTP {e.code}: {e.read().decode('utf-8', errors='replace')[:200]}"}, 0.0, None
        except Exception as e:
            return {"error": str(e)}, 0.0, None

    def is_ready(self) -> bool:
        """检查服务是否就绪"""
        try:
            req = urllib.request.Request(f"{self.base_url}/health")
            with urllib.request.urlopen(req, timeout=5) as resp:
                return resp.status == 200
        except Exception:
            return False


# ============================================================
# 文本处理工具
# ============================================================

def strip_thinking(text: str) -> str:
    """剥离 Qwen3 thinking 模式的 <think>...</think> 内容"""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text


def extract_text(response: dict) -> str:
    """从 API 响应中提取文本"""
    if "error" in response:
        return f"[ERROR: {response['error']}]"
    choices = response.get("choices", [])
    if not choices:
        return "[NO CHOICES]"
    text = choices[0].get("message", {}).get("content", "")
    return strip_thinking(text)


def extract_gsm8k_answer(text: str) -> Optional[str]:
    """从 GSM8K 输出中提取数值答案"""
    def normalize(num_str: str) -> str:
        num_str = num_str.replace(",", "").replace("$", "")
        try:
            val = float(num_str)
            return str(int(val)) if val == int(val) else str(val)
        except ValueError:
            return num_str

    # 优先匹配 #### 格式
    match = re.search(r"####\s*(\d+\.?\d*)", text)
    if match:
        return normalize(match.group(1))

    # 匹配 "answer is X" / "结果是X" 等
    match = re.search(
        r"(?:answer\s+is|result\s+is|答案是|结果是)[\s:]*\$?(\d+\.?\d*)",
        text, re.IGNORECASE,
    )
    if match:
        return normalize(match.group(1))

    # 最后一行的数字
    lines = text.strip().split("\n")
    for line in reversed(lines):
        match = re.search(r"(\d+\.?\d*)\s*\.?\s*$", line)
        if match:
            return normalize(match.group(1))

    # 文本中最后一个数字
    numbers = re.findall(r"\$?(\d+\.?\d*)", text)
    if numbers:
        return normalize(numbers[-1])

    return None


def extract_mmlu_answer(text: str) -> Optional[int]:
    """从 MMLU 输出中提取选项 (0=A, 1=B, 2=C, 3=D)"""
    text_upper = text.upper().strip()
    # 精确匹配 "Answer: A" 等
    for i, letter in enumerate(["A", "B", "C", "D"]):
        if re.search(rf"(?:answer|选择|选项)\s*(?:is|:|为)\s*{letter}\b", text_upper):
            return i
    # 匹配 "A." 开头
    for i, letter in enumerate(["A", "B", "C", "D"]):
        if re.match(rf"^{letter}[\.\)]", text_upper):
            return i
    # 单字母回答
    if text_upper in ["A", "B", "C", "D"]:
        return ["A", "B", "C", "D"].index(text_upper)
    # 最后尝试: 文本中出现的第一个字母
    for i, letter in enumerate(["A", "B", "C", "D"]):
        if letter in text_upper:
            return i
    return None


def extract_code(text: str) -> str:
    """从模型输出中提取 Python 代码"""
    # 优先: ```python ... ```
    match = re.search(r"```python\s*(.*?)\s*```", text, re.DOTALL)
    if match:
        return match.group(1)
    # 次选: ``` ... ```
    match = re.search(r"```\s*(.*?)\s*```", text, re.DOTALL)
    if match:
        return match.group(1)
    # 尝试: def 开头
    match = re.search(r"(def\s+\w+.*?)(?=\n\n|\Z)", text, re.DOTALL)
    if match:
        return match.group(1)
    return text


# ============================================================
# 数据加载
# ============================================================

def load_samples(filepath: str) -> List[Dict]:
    """从 JSON 文件加载样本"""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


# ============================================================
# 评估器
# ============================================================

class Evaluator:
    """评估器基类"""

    def __init__(self, clients: List[LLMClient], num_samples: int = 0):
        self.clients = clients
        self.num_samples = num_samples

    def evaluate(self, samples: List[Dict]) -> Dict:
        raise NotImplementedError

    def _limit_samples(self, samples: List[Dict]) -> List[Dict]:
        if self.num_samples > 0:
            return samples[: self.num_samples]
        return samples

    def _print_progress(self, current: int, total: int, step: int = 50):
        if (current + 1) % step == 0 or current + 1 == total:
            print(f"  进度: {current + 1}/{total}")


class GSM8KEvaluator(Evaluator):
    """GSM8K 数学推理评估"""

    def evaluate(self, samples: List[Dict]) -> Dict:
        samples = self._limit_samples(samples)
        n = len(samples)
        print(f"\n{'=' * 70}")
        print(f"GSM8K 数学推理评估 ({n} 样本)")
        print(f"{'=' * 70}")

        results = {
            c.name: {"correct": 0, "total": 0, "latencies": [], "details": []}
            for c in self.clients
        }

        for i, sample in enumerate(samples):
            self._print_progress(i, n, 50)
            prompt = (
                f"Solve this math problem step by step. "
                f"Give the final numerical answer after '####'.\n\n"
                f"Question: {sample['question']}\n\nAnswer:"
            )
            expected = str(sample["answer"]).strip()

            for client in self.clients:
                resp, latency, _ = client.chat(
                    [{"role": "user", "content": prompt}],
                    max_tokens=512, temperature=0.0,
                )
                text = extract_text(resp)
                predicted = extract_gsm8k_answer(text)
                correct = predicted == expected

                results[client.name]["correct"] += int(correct)
                results[client.name]["total"] += 1
                if latency > 0:
                    results[client.name]["latencies"].append(latency)
                results[client.name]["details"].append({
                    "question": sample["question"][:100],
                    "expected": expected,
                    "predicted": predicted,
                    "correct": correct,
                    "latency": latency,
                })

        return self._summarize(results)

    def _summarize(self, results: Dict) -> Dict:
        summary = {}
        for name, data in results.items():
            acc = data["correct"] / data["total"] * 100 if data["total"] else 0
            lats = data["latencies"]
            avg_lat = statistics.mean(lats) if lats else 0
            summary[name] = {
                "accuracy": acc, "correct": data["correct"], "total": data["total"],
                "avg_latency": avg_lat,
                "p50_latency": statistics.median(lats) if lats else 0,
                "p95_latency": _percentile(lats, 95) if lats else 0,
            }
            print(f"  {name}: {data['correct']}/{data['total']} ({acc:.1f}%), "
                  f"avg {avg_lat:.2f}s, p95 {_percentile(lats, 95):.2f}s")
        return {"summary": summary, "details": {n: d["details"] for n, d in results.items()}}


class MMLUEvaluator(Evaluator):
    """MMLU 多任务语言理解评估"""

    def evaluate(self, samples: List[Dict]) -> Dict:
        samples = self._limit_samples(samples)
        n = len(samples)
        print(f"\n{'=' * 70}")
        print(f"MMLU 多任务语言理解评估 ({n} 样本)")
        print(f"{'=' * 70}")

        results = {
            c.name: {"correct": 0, "total": 0, "latencies": [], "details": []}
            for c in self.clients
        }

        for i, sample in enumerate(samples):
            self._print_progress(i, n, 50)
            choices = sample["choices"]
            choices_text = "\n".join(
                f"{'ABCD'[j]}. {c}" for j, c in enumerate(choices)
            )
            prompt = (
                f"Answer the following multiple choice question. "
                f"Give only the letter (A/B/C/D).\n\n"
                f"Question: {sample['question']}\n\nChoices:\n{choices_text}\n\nAnswer:"
            )
            expected = sample["answer"]

            for client in self.clients:
                resp, latency, _ = client.chat(
                    [{"role": "user", "content": prompt}],
                    max_tokens=128, temperature=0.0,
                )
                text = extract_text(resp)
                predicted = extract_mmlu_answer(text)
                correct = predicted == expected

                results[client.name]["correct"] += int(correct)
                results[client.name]["total"] += 1
                if latency > 0:
                    results[client.name]["latencies"].append(latency)
                results[client.name]["details"].append({
                    "question": sample["question"][:100],
                    "expected": "ABCD"[expected],
                    "predicted": "ABCD"[predicted] if predicted is not None else None,
                    "correct": correct,
                    "latency": latency,
                })

        return self._summarize(results)

    def _summarize(self, results: Dict) -> Dict:
        summary = {}
        for name, data in results.items():
            acc = data["correct"] / data["total"] * 100 if data["total"] else 0
            lats = data["latencies"]
            avg_lat = statistics.mean(lats) if lats else 0
            summary[name] = {
                "accuracy": acc, "correct": data["correct"], "total": data["total"],
                "avg_latency": avg_lat,
                "p50_latency": statistics.median(lats) if lats else 0,
                "p95_latency": _percentile(lats, 95) if lats else 0,
            }
            print(f"  {name}: {data['correct']}/{data['total']} ({acc:.1f}%), "
                  f"avg {avg_lat:.2f}s")
        return {"summary": summary, "details": {n: d["details"] for n, d in results.items()}}


class HumanEvalEvaluator(Evaluator):
    """HumanEval 代码生成评估 (执行测试用例)"""

    def evaluate(self, samples: List[Dict]) -> Dict:
        samples = self._limit_samples(samples)
        n = len(samples)
        print(f"\n{'=' * 70}")
        print(f"HumanEval 代码生成评估 ({n} 样本)")
        print(f"{'=' * 70}")

        results = {
            c.name: {"passed": 0, "total": 0, "latencies": [], "details": []}
            for c in self.clients
        }

        for i, sample in enumerate(samples):
            self._print_progress(i, n, 20)
            prompt = (
                f"Complete the following Python function. "
                f"Return only the code, no explanation.\n\n{sample['prompt']}"
            )

            for client in self.clients:
                resp, latency, _ = client.chat(
                    [{"role": "user", "content": prompt}],
                    max_tokens=512, temperature=0.0,
                )
                text = extract_text(resp)
                code = extract_code(text)

                passed, error_msg = self._run_test(sample, code)

                results[client.name]["passed"] += int(passed)
                results[client.name]["total"] += 1
                if latency > 0:
                    results[client.name]["latencies"].append(latency)
                results[client.name]["details"].append({
                    "task_id": sample["task_id"],
                    "passed": passed,
                    "error": error_msg,
                    "latency": latency,
                    "code_preview": code[:200],
                })

        return self._summarize(results)

    def _run_test(self, sample: Dict, code: str) -> Tuple[bool, str]:
        """执行测试用例"""
        try:
            full_code = sample["prompt"] + code + "\n" + sample["test"]
            exec_globals = {}
            exec(full_code, exec_globals)
            test_func = exec_globals.get(f"check_{sample['entry_point']}")
            if test_func:
                test_func()
                return True, ""
            exec(f"check_{sample['entry_point']}()", exec_globals)
            return True, ""
        except Exception as e:
            return False, str(e)[:200]

    def _summarize(self, results: Dict) -> Dict:
        summary = {}
        for name, data in results.items():
            rate = data["passed"] / data["total"] * 100 if data["total"] else 0
            lats = data["latencies"]
            avg_lat = statistics.mean(lats) if lats else 0
            summary[name] = {
                "pass_rate": rate, "passed": data["passed"], "total": data["total"],
                "avg_latency": avg_lat,
            }
            print(f"  {name}: {data['passed']}/{data['total']} ({rate:.1f}%), "
                  f"avg {avg_lat:.2f}s")
        return {"summary": summary, "details": {n: d["details"] for n, d in results.items()}}


# ============================================================
# 吞吐量测试
# ============================================================

def test_throughput(client: LLMClient, num_requests: int = 50) -> Dict:
    """吞吐量测试: 固定 prompt 循环请求"""
    print(f"\n{'=' * 70}")
    print(f"{client.name} 吞吐量测试 ({num_requests} 请求)")
    print(f"{'=' * 70}")

    prompts = [
        "Write a Python function to compute fibonacci numbers.",
        "Explain the difference between TCP and UDP in networking.",
        "What is speculative decoding in LLM inference?",
        "Describe the architecture of a transformer model.",
        "How does gradient descent work in machine learning?",
    ]

    total_tokens = 0
    total_latency = 0.0
    latencies = []

    for i in range(num_requests):
        prompt = prompts[i % len(prompts)]
        resp, latency, _ = client.chat(
            [{"role": "user", "content": prompt}],
            max_tokens=256, temperature=0.0,
        )
        if "error" not in resp:
            tokens = resp.get("usage", {}).get("completion_tokens", 0)
            total_tokens += tokens
            total_latency += latency
            latencies.append(latency)
            if (i + 1) % 10 == 0:
                print(f"  进度: {i + 1}/{num_requests}, 当前延迟: {latency:.2f}s")

    avg_latency = statistics.mean(latencies) if latencies else 0
    throughput = total_tokens / total_latency if total_latency > 0 else 0

    print(f"\n  {client.name} 吞吐量结果:")
    print(f"    总请求: {len(latencies)}")
    print(f"    总 tokens: {total_tokens}")
    print(f"    总延迟: {total_latency:.2f}s")
    print(f"    平均延迟: {avg_latency:.2f}s")
    print(f"    P95 延迟: {_percentile(latencies, 95):.2f}s")
    print(f"    吞吐量: {throughput:.2f} tok/s")

    return {
        "total_requests": len(latencies),
        "total_tokens": total_tokens,
        "total_latency": total_latency,
        "avg_latency": avg_latency,
        "p95_latency": _percentile(latencies, 95),
        "throughput_toks": throughput,
    }


def test_ttft(client: LLMClient, num_requests: int = 20) -> Dict:
    """TTFT (Time To First Token) 测试: 流式请求"""
    print(f"\n{'=' * 70}")
    print(f"{client.name} TTFT 测试 ({num_requests} 请求, 流式)")
    print(f"{'=' * 70}")

    prompt = "Explain what artificial intelligence is in detail."
    ttfts = []
    total_lats = []

    for i in range(num_requests):
        resp, latency, ttft = client.chat(
            [{"role": "user", "content": prompt}],
            max_tokens=256, temperature=0.0, stream=True,
        )
        if ttft is not None:
            ttfts.append(ttft)
            total_lats.append(latency)
        if (i + 1) % 5 == 0:
            print(f"  进度: {i + 1}/{num_requests}, TTFT: {ttft:.3f}s")

    avg_ttft = statistics.mean(ttfts) if ttfts else 0
    p95_ttft = _percentile(ttfts, 95) if ttfts else 0
    avg_total = statistics.mean(total_lats) if total_lats else 0

    print(f"\n  {client.name} TTFT 结果:")
    print(f"    平均 TTFT: {avg_ttft:.3f}s")
    print(f"    P95 TTFT: {p95_ttft:.3f}s")
    print(f"    平均总延迟: {avg_total:.2f}s")

    return {
        "avg_ttft": avg_ttft,
        "p95_ttft": p95_ttft,
        "avg_total_latency": avg_total,
        "num_requests": len(ttfts),
    }


# ============================================================
# 工具函数
# ============================================================

def _percentile(data: List[float], p: float) -> float:
    """计算百分位数"""
    if not data:
        return 0.0
    sorted_data = sorted(data)
    k = (len(sorted_data) - 1) * p / 100
    f = int(k)
    c = f + 1 if f + 1 < len(sorted_data) else f
    return sorted_data[f] + (sorted_data[c] - sorted_data[f]) * (k - f)


# ============================================================
# 报告生成
# ============================================================

def generate_markdown_report(results: Dict, output_path: str):
    """生成 Markdown 汇总报告"""
    lines = [
        "# 推测解码评估报告",
        "",
        f"测试时间: {results['timestamp']}",
        f"模型: {results.get('model', 'N/A')}",
        "",
        "## 精度对比",
        "",
    ]

    for dataset_name, eval_data in results.get("evaluations", {}).items():
        lines.append(f"### {dataset_name.upper()}")
        lines.append("")
        if dataset_name == "humaneval":
            lines.append("| 服务 | 通过数/总数 | 通过率 | 平均延迟 | P95延迟 |")
            lines.append("|------|-----------|--------|---------|---------|")
            for name, s in eval_data["summary"].items():
                lines.append(
                    f"| {name} | {s['passed']}/{s['total']} | {s['pass_rate']:.1f}% | "
                    f"{s['avg_latency']:.2f}s | {s.get('p95_latency', 0):.2f}s |"
                )
        else:
            lines.append("| 服务 | 正确数/总数 | 准确率 | 平均延迟 | P50 | P95 |")
            lines.append("|------|-----------|--------|---------|-----|-----|")
            for name, s in eval_data["summary"].items():
                lines.append(
                    f"| {name} | {s['correct']}/{s['total']} | {s['accuracy']:.1f}% | "
                    f"{s['avg_latency']:.2f}s | {s.get('p50_latency', 0):.2f}s | "
                    f"{s.get('p95_latency', 0):.2f}s |"
                )
        lines.append("")

    if results.get("throughput"):
        lines.append("## 吞吐量对比")
        lines.append("")
        lines.append("| 服务 | 总请求 | 总tokens | 总延迟 | 平均延迟 | P95 | 吞吐量 |")
        lines.append("|------|--------|---------|--------|---------|-----|--------|")
        for name, t in results["throughput"].items():
            lines.append(
                f"| {name} | {t['total_requests']} | {t['total_tokens']} | "
                f"{t['total_latency']:.2f}s | {t['avg_latency']:.2f}s | "
                f"{t.get('p95_latency', 0):.2f}s | {t['throughput_toks']:.2f} tok/s |"
            )
        lines.append("")

    if results.get("ttft"):
        lines.append("## TTFT (首 Token 延迟)")
        lines.append("")
        lines.append("| 服务 | 平均TTFT | P95 TTFT | 平均总延迟 |")
        lines.append("|------|---------|---------|-----------|")
        for name, t in results["ttft"].items():
            lines.append(
                f"| {name} | {t['avg_ttft']:.3f}s | {t['p95_ttft']:.3f}s | "
                f"{t['avg_total_latency']:.2f}s |"
            )
        lines.append("")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Markdown 报告已保存: {output_path}")


# ============================================================
# 主函数
# ============================================================

def main():
    parser = argparse.ArgumentParser(
        description="推测解码评估参考脚本",
        formatter_class=argparse.RawDescriptionHelpFormatter,
    )
    parser.add_argument("--dspark-url", default="http://127.0.0.1:8000")
    parser.add_argument("--dflash-url", default="http://127.0.0.1:8001")
    parser.add_argument("--baseline-url", help="Baseline 服务 URL (无推测解码)")
    parser.add_argument("--data-dir", default="./eval_data", help="数据文件目录")
    parser.add_argument(
        "--dataset", default="all",
        choices=["all", "gsm8k", "mmlu", "humaneval"],
    )
    parser.add_argument("--num-samples", type=int, default=0, help="限制样本数 (0=全部)")
    parser.add_argument("--test-throughput", action="store_true", help="测试吞吐量")
    parser.add_argument("--test-ttft", action="store_true", help="测试 TTFT (流式)")
    parser.add_argument("--num-throughput-requests", type=int, default=50)
    parser.add_argument("--output-dir", default="./eval_results")
    args = parser.parse_args()

    # 构建客户端列表
    clients = []
    service_configs = [
        ("DSpark", args.dspark_url),
        ("DFlash", args.dflash_url),
        ("Baseline", args.baseline_url),
    ]
    for name, url in service_configs:
        if not url:
            continue
        print(f"连接 {name}: {url} ...")
        client = LLMClient(url, name)
        if client.model and client.is_ready():
            clients.append(client)
            print(f"  ✓ 就绪, 模型: {client.model}")
        else:
            print(f"  ✗ 无法连接或服务未就绪")

    if not clients:
        print("\n[ERROR] 没有可用的服务")
        sys.exit(1)

    print(f"\n{'=' * 70}")
    print(f"推测解码评估")
    print(f"{'=' * 70}")
    print(f"服务: {[c.name for c in clients]}")
    print(f"数据集: {args.dataset}")
    print(f"吞吐量测试: {args.test_throughput}")
    print(f"TTFT 测试: {args.test_ttft}")

    results = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model": clients[0].model if clients else "N/A",
        "config": {
            "services": {c.name: c.base_url for c in clients},
            "data_dir": args.data_dir,
            "dataset": args.dataset,
            "num_samples": args.num_samples,
        },
        "evaluations": {},
        "throughput": {},
        "ttft": {},
    }

    # 精度评估
    if args.dataset in ("all", "gsm8k"):
        path = os.path.join(args.data_dir, "gsm8k_test_250.json")
        if os.path.exists(path):
            samples = load_samples(path)
            evaluator = GSM8KEvaluator(clients, args.num_samples)
            results["evaluations"]["gsm8k"] = evaluator.evaluate(samples)
        else:
            print(f"[WARN] GSM8K 数据不存在: {path}")

    if args.dataset in ("all", "mmlu"):
        path = os.path.join(args.data_dir, "mmlu_test_250.json")
        if os.path.exists(path):
            samples = load_samples(path)
            evaluator = MMLUEvaluator(clients, args.num_samples)
            results["evaluations"]["mmlu"] = evaluator.evaluate(samples)
        else:
            print(f"[WARN] MMLU 数据不存在: {path}")

    if args.dataset in ("all", "humaneval"):
        path = os.path.join(args.data_dir, "humaneval_test_164.json")
        if os.path.exists(path):
            samples = load_samples(path)
            evaluator = HumanEvalEvaluator(clients, args.num_samples)
            results["evaluations"]["humaneval"] = evaluator.evaluate(samples)
        else:
            print(f"[WARN] HumanEval 数据不存在: {path}")

    # 吞吐量测试
    if args.test_throughput:
        for client in clients:
            results["throughput"][client.name] = test_throughput(
                client, args.num_throughput_requests
            )

    # TTFT 测试
    if args.test_ttft:
        for client in clients:
            results["ttft"][client.name] = test_ttft(client, num_requests=20)

    # 汇总
    print(f"\n{'=' * 70}")
    print(f"总体评估结果")
    print(f"{'=' * 70}")
    for ds_name, eval_data in results["evaluations"].items():
        print(f"\n{ds_name.upper()}:")
        for name, s in eval_data["summary"].items():
            if ds_name == "humaneval":
                print(f"  {name}: {s['pass_rate']:.1f}%")
            else:
                print(f"  {name}: {s['accuracy']:.1f}%")
    if results["throughput"]:
        print(f"\n吞吐量:")
        for name, t in results["throughput"].items():
            print(f"  {name}: {t['throughput_toks']:.2f} tok/s")
    if results["ttft"]:
        print(f"\nTTFT:")
        for name, t in results["ttft"].items():
            print(f"  {name}: {t['avg_ttft']:.3f}s")

    # 保存报告
    os.makedirs(args.output_dir, exist_ok=True)
    ts = int(time.time())
    json_path = os.path.join(args.output_dir, f"eval_report_{ts}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\nJSON 报告已保存: {json_path}")

    md_path = os.path.join(args.output_dir, f"eval_report_{ts}.md")
    generate_markdown_report(results, md_path)

    print(f"\n评估完成")


if __name__ == "__main__":
    main()

## 2.3 部署命令

推理引擎为vLLM 0.26.0（镜像vllm/vllm-openai:latest）。

启动参考：

```
docker run -d --name dspark-test \
  --gpus '"device=0,1,2,3"' \
  --shm-size=16g \
  -p 8000:8000 \
  -v /root/local_models:/models \
  vllm/vllm-openai:latest \
  --model /models/Qwen3-4B \
  --speculative-config '{"method":"dspark","model":"/models/dspark_qwen3_4b_block7","num_speculative_tokens":4}' \
  --dtype bfloat16 \
  --tensor-parallel-size 4 \
  --port 8000
```
